# LlamaIndex Practice

Welcome to your hands-on practice with LlamaIndex! In this notebook, you'll learn how to:
- Create custom tools for AI agents
- Build a functional agent using Azure OpenAI
- Integrate Model Context Protocol (MCP) tools
- Implement observability and debugging
- Complete a creative challenge to apply your knowledge

## What You'll Build

In this notebook, you'll create an intelligent **Travel Planning Assistant** that can:
- Search for flight information
- Get weather forecasts for destinations
- Convert currencies for travel budgeting
- Use Model Context Protocol (MCP) for enhanced capabilities

By the end, you'll understand the complete agent creation flow with LlamaIndex and be ready to build your own creative agents!

## Step 1: Initial Setup & Installation

First, let's install the required packages. We need:
- **llama-index-core**: Core LlamaIndex framework (agents, tools, workflows)
- **llama-index-llms-openai**: OpenAI-compatible LLM integration (works with Azure OpenAI's v1 endpoint)
- **python-dotenv**: Loads credentials from a `.env` file

In [3]:
%pip install -q llama-index-core llama-index-llms-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 2: Configure Azure OpenAI Credentials

You'll need an Azure OpenAI API key, endpoint, and model deployment. Store them in a `.env` file (in this folder or any parent folder); if a value is missing, the cell below prompts for it.
```
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_ENDPOINT=https://<resource>.openai.azure.com/openai/v1/
AZURE_OPENAI_DEPLOYMENT=gpt-5.4-mini
```

In [4]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.6-luna


## Step 3: Create Custom Tools

Tools are functions that your agent can call to perform specific tasks. Let's create travel-related tools.

**Key Concepts:**
- Each tool is a Python function with clear docstrings
- Type hints help the LLM understand parameter types
- Docstrings describe what the tool does (the LLM reads this!)

In [5]:
from datetime import datetime, timedelta
import random


def search_flights(origin: str, destination: str, date: str) -> str:
    """Search for available flights between two cities on a specific date.

    Args:
        origin: The departure city (e.g., 'New York')
        destination: The arrival city (e.g., 'Paris')
        date: The travel date in YYYY-MM-DD format

    Returns:
        A string with flight information including price and duration
    """
    # Simulate flight search
    flight_number = f"FL{random.randint(100, 999)}"
    price = random.randint(300, 1500)
    duration = random.randint(4, 15)

    return f"Found flight {flight_number} from {origin} to {destination} on {date}. Price: ${price}, Duration: {duration}h"


def get_weather(city: str, days_ahead: int = 0) -> str:
    """Get weather forecast for a city.

    Args:
        city: The city name to get weather for
        days_ahead: Number of days ahead to forecast (0 = today, 1 = tomorrow, etc.)

    Returns:
        A string describing the weather conditions
    """
    conditions = ["Sunny", "Partly Cloudy", "Cloudy", "Rainy", "Stormy"]
    condition = random.choice(conditions)
    temp = random.randint(15, 30)
    forecast_date = datetime.now() + timedelta(days=days_ahead)

    return f"Weather in {city} on {forecast_date.strftime('%Y-%m-%d')}: {condition}, {temp}°C"


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert
        from_currency: Source currency code (e.g., 'USD')
        to_currency: Target currency code (e.g., 'EUR')

    Returns:
        A string with the converted amount
    """
    # Simplified exchange rates (in practice, you'd use a real API)
    rates = {
        ("USD", "EUR"): 0.92,
        ("USD", "GBP"): 0.79,
        ("USD", "JPY"): 149.50,
        ("EUR", "USD"): 1.09,
        ("GBP", "USD"): 1.27,
    }

    rate = rates.get((from_currency, to_currency), 1.0)
    converted = amount * rate

    return f"{amount} {from_currency} = {converted:.2f} {to_currency}"

## Step 4: Initialize the Language Model

Now let's set up Azure OpenAI as our LLM. Azure OpenAI's v1 API is OpenAI-compatible, so we use LlamaIndex's `OpenAI` class with `api_base` pointing at the Azure endpoint.

**Note:** `model` is your deployment name. It must match an OpenAI model name (e.g. `gpt-5.4-mini`, `gpt-5.6-luna`) so LlamaIndex knows the model's context window and tool-calling support.

In [6]:
from llama_index.llms.openai import OpenAI

llm = OpenAI(
    model=AZURE_OPENAI_DEPLOYMENT,
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

## Step 5: Create Your First Agent

A **FunctionAgent** is LlamaIndex's agentic workflow that can:
1. Understand user queries
2. Decide which tools to use
3. Call tools with appropriate parameters
4. Synthesize responses from tool outputs

Let's create our Travel Planning Agent!

In [7]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm,
    verbose=True  # This helps us see what the agent is doing
)

## Step 6: Create an Observability Helper

Before testing our agent, let's create a helper function that gives us **detailed visibility** into what the agent is doing.

**Why Observability Matters:**
- 🔍 See which tools the agent chooses
- 📊 Inspect the inputs passed to each tool
- ✅ Verify the outputs returned
- 🐛 Debug issues when the agent doesn't behave as expected

The `run_agent_verbose()` function will:
1. Display the user's query
2. Stream events as the agent works
3. Print details about each tool call
4. Return the final response

This is crucial for understanding and debugging agent behavior!

In [8]:
from llama_index.core.agent.workflow import ToolCallResult


async def run_agent_verbose(query: str):
    """Run the agent and print detailed information about tool calls."""
    print(f"🤖 User Query: {query}\n")
    print("=" * 60)

    handler = agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 Tool Used: {event.tool_name}")
            print(f"📥 Input: {event.tool_kwargs}")
            print(f"📤 Output: {event.tool_output}")
            print("-" * 60)

    result = await handler
    print(f"\n✨ Final Response:\n{result}\n")
    return result

## Step 7: Test Your Agent with Simple Queries

Let's start with basic queries to see how the agent uses tools:

In [9]:
response = await run_agent_verbose(
    "What's the weather like in Tokyo today?"
)

🤖 User Query: What's the weather like in Tokyo today?

[tick] add: AgentWorkflowStartEvent(user_msg="What's the weather like in Tokyo today?", chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="What's the weather like in Tokyo today?")])], current_agent_name='Age...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, addit

In [10]:
# Inspect which tools were called
print("Tool Calls Made:")
for tool_call in response.tool_calls:
    print(f"  - {tool_call}")

Tool Calls Made:
  - tool_name='get_weather' tool_kwargs={'city': 'Tokyo', 'days_ahead': 0} tool_id='call_uDWd6nRacmeGM4iNkYivFOFx' tool_output=ToolOutput(blocks=[TextBlock(block_type='text', text='Weather in Tokyo on 2026-11-23: Stormy, 18°C')], tool_name='get_weather', raw_input={'args': (), 'kwargs': {'city': 'Tokyo', 'days_ahead': 0}}, raw_output='Weather in Tokyo on 2026-11-23: Stormy, 18°C', is_error=False) return_direct=False


## Step 8: Complex Multi-Tool Query

Watch how the agent chains multiple tools together to answer complex questions:

In [11]:
response = await run_agent_verbose(
    "I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?"
)

🤖 User Query: I want to fly from London to Barcelona on 2026-12-15. What's the weather there, and how much is 500 USD in EUR?

[tick] add: AgentWorkflowStartEvent(user_msg='I want to fly from London to Barcelona on 2026-12-...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="I want to fly from London to Barcelona on 2026-12-15. What's the weat...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] 

## Step 9: Managing Context & Memory

By default, `.run()` is stateless (doesn't remember previous conversations). Let's add memory using a **Context** object:

In [12]:
from llama_index.core.workflow import Context

# Create a new agent instance for this demo
memory_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency],
    llm=llm
)

# Create a context to maintain conversation history
ctx = Context(memory_agent)

# First message - introduce yourself
response1 = await memory_agent.run(
    "Hi! I'm planning a trip to Rome in December. My name is Alex.",
    ctx=ctx
)
print(f"Response 1: {response1}\n")

# Second message - agent should remember your name
response2 = await memory_agent.run(
    "What's my name, and what's the weather like in my destination?",
    ctx=ctx
)
print(f"Response 2: {response2}")

Response 1: Hi Alex! Rome in December sounds wonderful—fewer crowds, festive lights, and plenty of history and food to enjoy. I can help with flights, weather, accommodation ideas, or a day-by-day itinerary. What would you like to plan first?



c:\Users\artis.fribergs\AppData\Local\Programs\Python\Python314\Lib\site-packages\workflows\context\state_store.py:231: UnserializableKeyWarning: Skipping serialization of known unserializable key: memory -- This is expected but will require this item to be set manually after deserialization.
  warnings.warn(


Response 2: I don’t know your name or destination yet. What name should I use, and which destination should I check the weather for?


## Step 10: Advanced - Model Context Protocol (MCP) Tools

MCP is a standardized protocol for connecting AI models to external tools and data sources. Let's integrate an MCP tool!

**What is MCP?**
- Open protocol for AI-to-tool communication
- Enables standardized tool integration
- Supports local and remote tools

### Creating a Custom MCP-Style Tool

For this practice, we simulate an MCP tool that fetches travel tips with a regular `FunctionTool`.

To load tools from a real MCP server, install `llama-index-tools-mcp` and convert the server's tools into LlamaIndex tools:
```python
# from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
# mcp_client = BasicMCPClient("http://127.0.0.1:8000/mcp")  # or BasicMCPClient("python", args=["server.py"]) for stdio
# mcp_tools = await McpToolSpec(client=mcp_client).to_tool_list_async()
```

In [13]:
from llama_index.core.tools import FunctionTool


def get_travel_tips(destination: str) -> str:
    """Get essential travel tips for a destination.

    Args:
        destination: The city or country to get tips for

    Returns:
        A string with helpful travel tips
    """
    tips_database = {
        "paris": "🇫🇷 Tips for Paris: Try croissants at local boulangeries, use the Metro, visit Eiffel Tower early morning",
        "tokyo": "🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa",
        "new york": "🇺🇸 Tips for NYC: Use subway, try pizza and bagels, visit Central Park and Times Square",
        "barcelona": "🇪🇸 Tips for Barcelona: Visit Sagrada Familia, try tapas and paella, walk Las Ramblas",
        "rome": "🇮🇹 Tips for Rome: Visit Colosseum early, try authentic carbonara, throw coin in Trevi Fountain",
    }

    city_lower = destination.lower()
    for key in tips_database:
        if key in city_lower:
            return tips_database[key]

    return f"💡 General tips: Research local customs, learn basic phrases, try local cuisine!"


# Convert to LlamaIndex tool
travel_tips_tool = FunctionTool.from_defaults(fn=get_travel_tips)

# Create agent with MCP-style tool
mcp_agent = FunctionAgent(
    tools=[search_flights, get_weather, convert_currency, travel_tips_tool],
    llm=llm
)

print("✅ Agent with MCP tool created!")

✅ Agent with MCP tool created!


### Testing the Enhanced Agent

In [14]:
async def run_mcp_agent(query: str):
    """Run the MCP-enhanced agent."""
    print(f"🤖 Query: {query}\n")
    handler = mcp_agent.run(query)
    async for event in handler.stream_events():
        if isinstance(event, ToolCallResult):
            print(f"🔧 {event.tool_name} → {event.tool_output}")

    result = await handler
    print(f"\n✨ {result}\n")
    return result


response = await run_mcp_agent(
    "I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow."
)

🤖 Query: I'm planning a trip to Tokyo. Give me travel tips and check the weather for tomorrow.

🔧 get_travel_tips → 🇯🇵 Tips for Tokyo: Get a Suica card for transport, try ramen at local shops, visit temples in Asakusa
🔧 get_weather → Weather in Tokyo on 2026-11-24: Cloudy, 23°C

✨ ## Tokyo travel tips
- Get a **Suica or PASMO** card for trains, subways, and convenience-store purchases.
- Tokyo’s transit system is extensive—use **Google Maps** for routes and platform information.
- Visit **Asakusa** and Senso-ji Temple, explore **Shibuya** and **Shinjuku**, and consider quieter neighborhoods such as **Yanaka**.
- Try ramen at smaller local shops, along with sushi, izakaya dishes, and convenience-store snacks.
- Carry some cash, especially for smaller restaurants and temples.
- Be mindful of local etiquette: keep voices low on trains, queue properly, and avoid eating while walking in crowded areas.

## Tomorrow’s weather
**Tuesday, November 24, 2026:** Cloudy, around **23°C**.



## Step 11: Enhanced Observability with Custom Callbacks

Let's implement more detailed observability to track agent performance:

In [15]:
import time
from typing import List


class AgentObserver:
    """Track and display agent performance metrics."""

    def __init__(self):
        self.tool_calls = []
        self.start_time = None
        self.end_time = None

    async def observe_agent(self, query: str, agent: FunctionAgent):
        """Run agent and collect observability data."""
        self.tool_calls = []
        self.start_time = time.time()

        print(f"🎯 Query: {query}\n")
        print("📊 Observability Dashboard")
        print("=" * 60)

        handler = agent.run(query)
        async for event in handler.stream_events():
            if isinstance(event, ToolCallResult):
                self.tool_calls.append({
                    'tool': event.tool_name,
                    'input': event.tool_kwargs,
                    'output': event.tool_output
                })
                print(f"⚙️  Tool Call #{len(self.tool_calls)}: {event.tool_name}")

        result = await handler
        self.end_time = time.time()

        # Display metrics
        print("\n" + "=" * 60)
        print("📈 Performance Metrics:")
        print(f"  ⏱️  Total Time: {self.end_time - self.start_time:.2f}s")
        print(f"  🔧 Tools Called: {len(self.tool_calls)}")
        print(f"  📝 Tool Names: {[t['tool'] for t in self.tool_calls]}")
        print(f"\n💬 Response: {result}")

        return result


# Test the observer
observer = AgentObserver()
await observer.observe_agent(
    "Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY",
    mcp_agent
)

🎯 Query: Find flights from Paris to Tokyo on 2026-12-20, check weather, and convert 1000 EUR to JPY

📊 Observability Dashboard
⚙️  Tool Call #1: convert_currency
⚙️  Tool Call #2: search_flights
⚙️  Tool Call #3: get_weather

📈 Performance Metrics:
  ⏱️  Total Time: 2.72s
  🔧 Tools Called: 3
  📝 Tool Names: ['convert_currency', 'search_flights', 'get_weather']

💬 Response: - **Flight:** Paris → Tokyo on **December 20, 2026**  
  - Flight: **FL791**
  - Price: **$1,150**
  - Duration: **8 hours**

- **Tokyo weather:** Cloudy, **16°C**

- **Currency conversion:** **€1,000 = ¥1,000**


AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='- **Flight:** Paris → Tokyo on **December 20, 2026**  \n  - Flight: **FL791**\n  - Price: **$1,150**\n  - Duration: **8 hours**\n\n- **Tokyo weather:** Cloudy, **16°C**\n\n- **Currency conversion:** **€1,000 = ¥1,000**')]), structured_response=None, current_agent_name='Agent', raw={'id': 'chatcmpl-EnOg2dVJ4KXwi4WotMZH8VSHKWXkt', 'choices': [{'delta': {'audio': None, 'content': None, 'function_call': None, 'refusal': None, 'role': None, 'tool_calls': None}, 'index': 0, 'finish_reason': 'stop', 'logprobs': None, 'content_filter_results': {}}], 'created': 1795441638, 'model': 'gpt-5.6-luna-2026-07-09', 'object': 'chat.completion.chunk', 'moderation': None, 'obfuscation': 'qifrKXDrkGiT', 'service_tier': 'default', 'system_fingerprint': None, 'usage': None}, tool_calls=[ToolCallResult(tool_name='search_flights', tool_kwargs={'origin': 'Paris', 'destina

## 🎉 Congratulations!

You've learned how to:
- ✅ Create custom tools for AI agents
- ✅ Build functional agents with LlamaIndex
- ✅ Implement observability and debugging
- ✅ Manage conversation context and memory
- ✅ Integrate MCP-style tools
- ✅ Track agent performance metrics

### Share Your Work:
- Complete the challenge and share your restaurant agent
- Experiment with different tools and domains
- Try integrating real APIs for production use

Happy building! 🚀

## 🎓 Challenge: Build Your Own Agent!

Now it's your turn! Apply what you've learned to create a unique agent.

### Your Mission:
Create a **Restaurant Recommendation Agent** that helps users find and learn about restaurants.

### Requirements:
1. **Create at least 3 custom tools:**
   - `find_restaurants(cuisine: str, city: str) -> str`: Find restaurants by cuisine type
   - `get_restaurant_details(restaurant_name: str) -> str`: Get details like hours, price range
   - `make_reservation(restaurant_name: str, date: str, party_size: int) -> str`: Simulate booking

2. **Add a bonus tool** (get creative!):
   - Menu translator
   - Dietary restriction filter
   - Restaurant reviews summarizer
   - Distance calculator
   - Or your own idea!

3. **Implement:**
   - Agent with all your tools
   - Observability (print tool calls)
   - Context/memory for conversation
   - Test with at least 2 complex queries

4. **Bonus Points:**
   - Make it work with real APIs (like Yelp or Google Places)
   - Add error handling
   - Create a user-friendly interface
   - Implement caching for repeated queries

### Tips:
- Use clear docstrings - the LLM reads them!
- Test each tool individually first
- Start simple, then add complexity
- Use type hints for better tool understanding
- Think about real-world use cases

### Example Test Queries:
- "Find Italian restaurants in Boston and get details about the top one"
- "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people tomorrow"
- "Recommend a restaurant with outdoor seating, check if they're open now, and book a table"

### Your Solution Space

Write your code below:

In [16]:
from llama_index.core.agent.workflow import FunctionAgent, ToolCall, ToolCallResult
from llama_index.core.workflow import Context
# TODO: Define your restaurant tools here
def find_restaurants(cuisine: str, city: str) -> str:
    """Find restourants in a city by cuisine type.

    Args:
        cuisine: The type of cuisine(e.g., "Italian","Indian")
        city: the city to search in (e.g., 'New York')

    Returns:
        A string listing restourant names with rating
    """
    suffixes=["Trattoria", "Kitchen","Bistro","house"]
    names= [f"{city}{cuisine}{s}" for s in random.sample(suffixes,3)]
    results=[f"{n}(rating{round(random.uniform(3.8,4.9),1)}/5)" for n in names]
    return f"{cuisine} restourant in {city}:"+"; ".join(results)
    pass


def get_restaurant_details(restaurant_name: str) -> str:
    """Get details about a restourant such as hours, price range and setting.

    Args:
        restourant_name: The name of the restourant
    Returns:
        A string with opening hours, price range, and seating options
    """
    price= random.choice(["$","$$","$$$"])
    outdoor= random.choice(["available","not available"])
    return (
        f"{restaurant_name}: open 11:00-22:00 daily, price range {price}, "
        f"outdoor seating {outdoor}, vegetarian options available."
    )
    pass


def make_reservation(restaurant_name: str, date: str, party_size: int) -> str:
    """Make a table reservation at a restaurant.

    Args:
        restaurant_name: The name of the restaurant
        date: The reservation date in YYYY-MM-DD format
        party_size: The number of people in the party

    Returns:
        A confirmation message with a confirmation code
    """
    code = f"RES{random.randint(1000, 9999)}"
    return (
        f"Reservation confirmed at {restaurant_name} on {date} "
        f"for {party_size} people. Confirmation code: {code}"
    )
    pass


# TODO: Add your bonus tool
def check_dietary_options(restaurant_name: str, diet: str) -> str:
    """Check whether a restaurant can accommodate a dietary restriction.

    Args:
        restaurant_name: The name of the restaurant
        diet: The dietary need (e.g., 'vegetarian', 'vegan', 'gluten-free')

    Returns:
        A string saying whether the diet is supported and how
    """
    supported = random.random() > 0.2
    if supported:
        return f"{restaurant_name} offers a dedicated {diet} menu section."
    return f"{restaurant_name} has limited {diet} options; please call ahead."


# TODO: Create your agent
today = datetime.now().strftime("%Y-%m-%d")

restaurant_agent = FunctionAgent(
    tools=[find_restaurants, get_restaurant_details, make_reservation, check_dietary_options],
    llm=llm,
    system_prompt=(
        "You are a helpful restaurant recommendation assistant. "
        f"Today's date is {today}; use it to convert words like 'tomorrow' into YYYY-MM-DD dates."
    ),
    verbose=True,
)

restaurant_ctx = Context(restaurant_agent)



# TODO: Test your agent with complex queries
response1 = await restaurant_agent.run(
    "Find Italian restaurants in Boston and get details about the top one",
    ctx=restaurant_ctx,
)
print(f"Response 1: {response1}\n")

response2 = await restaurant_agent.run(
    "I'm vegetarian. Find a good restaurant in Seattle and make a reservation for 4 people tomorrow",
    ctx=restaurant_ctx,
)
print(f"Response 2: {response2}\n")

response3 = await restaurant_agent.run(
    "Recommend a restaurant with outdoor seating, check if they're open now, and book a table",
    ctx=restaurant_ctx,
)
print(f"Response 3: {response3}")


[tick] add: AgentWorkflowStartEvent(user_msg='Find Italian restaurants in Boston and get details...', chat_history=None, memory=None, max_iterations=None, early_stopping_method=None)
[init_run:0] started from AgentWorkflowStartEvent
[init_run:0] complete with AgentInput
[tick] add: AgentInput(input=[ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Find Italian restaurants in Boston and get details about the top one'...
[setup_agent:0] started from AgentInput
[setup_agent:0] complete with AgentSetup
[tick] add: AgentSetup(input=[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text="You are a helpful restaurant recommendation assistant. Today's da...
[run_agent_step:0] started from AgentSetup
[run_agent_step:0] complete with AgentOutput
[tick] add: AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={'tool_calls': [ChoiceDeltaT